# Toxic Comments Classification - SVM

#### Author: SAFAA ENNACIRI - ENNS11538307                                 
#### Research Director: M. HAKIM LOUNIS

Dataset: Toxic Comment Classification dataset is a dataset of comments from Wikipedia’s talk page edits.  It is avaialble at Kaggle (https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data). 


The comments are divided in two classes: toxic and normal.

# 1. Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle, re
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import LinearSVC

# 2. Data Preparation 

In [41]:
# Load training data
df = pd.read_csv('../../Data/train_set3.csv')
df.head()

,Tweet ID,Text,tox
0,1230482808457527297,BREAKING: Many leading Chinese scientists are ...,0.0
1,1244980592615116800,"If China can kill us, they can kill you too @A...",1.0
2,1244899032771604480,Fuck u Chi Na!!! Karma is coming soon !!! The ...,1.0
3,1244895530720710656,Some Chinese are horrible as fuck! #ChinaLiedP...,1.0
4,1244341105879142407,"@ScottGottliebMD you are a ""doctor"" and you be...",1.0


In [42]:
df.describe()

,Tweet ID,tox
count,2.104000e+03,2104.000000
mean,1.240002e+18,0.486217
std,6.711621e+15,0.499929
min,1.215597e+18,0.000000
25%,1.232951e+18,0.000000
50%,1.244263e+18,0.000000
75%,1.245405e+18,1.000000
max,1.251185e+18,1.000000


In [44]:
for x in range(2000):
    print(df['Text'][x])
    print ('*******')

BREAKING: Many leading Chinese scientists are starting to speak out and say COVID-19 (coronavirus) originated at a government bioweapons research facility in Wuhan, rather than the widely-held belief that it emerged from the city's Huanan seafood markets.  https://t.co/DsxwJSAox2 https://t.co/TbshN8eDni
*******
If China can kill us, they can kill you too @AngusCh89615720 @Heefg14 @Jothehkgirl @jonathanseual86 @SammY91318198 @marielSiviglia @op12381932972 @Makhk6 @AWMF5 @HMW79256809 @Senthil557 @wasteslave @valwonghk @SallyGunner0331 @bhikshu2007 @Mark48994372
*******
Fuck u Chi Na!!! Karma is coming soon !!! The CCP Virus 🦠 （virus origin wuhan）😡😡😡Poor child 😭😭😭🙏🙏🙏 Now, Wuhan pneumonia is still a very serious outbreak in China. But these scums are still to beat up dogs the street.
*******
Some Chinese are horrible as fuck! #ChinaLiedPeopleDie #BoycottChina #WuhanVirus
*******
@ScottGottliebMD you are a "doctor" and you believe that shit?  you chinese hack...  #ChineseVirus
*******
The r

I remember when we first hear about the corona virus we was all let damn them Chinese niggas is wild now look 😟😟😟
*******
@MeqdadLocation Fuck @HassanRouhani and @AmbChangHua #ChineseVirus
*******
You fucking evil pieces of shit @china #china #fuckchina #evilonearth
*******
@jling_lee @GaryPNabhan @HuXijin_GT Hi, chink Lee. A beauty, isn’t xi? https://t.co/V60dWgPDB0
*******
Weird shit inside the gold mine....Undercover Investigation Indicates Coronavirus Death Toll in Wuhan Much ... https://t.co/ck7f2HCMGa via @YouTube
*******
@LHGHKG @RyanAFournier @Aloha88617098 true "shit" #BoicoteAChina #chinazi  #StandWithHongKong #ChineseWuhanVirus #ChinaLiedAndPeopleDied #ChinaLiedPeopleDied #ChineseVirusCorona #chinaVirusMadeInChina #ChinaMustPay #ChinaWuhanVirus
*******
Only difference is that we don't have 60+ years of anti imperialism propaganda. Because it's the same thing the west loses its shit over China.
*******
Why everyone is remaining silent right now is a little alarming. Shitz ser

In [10]:
def c_text(comment): 
    comment = comment.lower() # Normalize to lowercase 

    if u'ô' in comment:
        print (comment)
        print("--------------------------------------------")
    return comment

In [4]:
# clean and normalize comments
df['Text'] = df['Text'].map(lambda word:c_text(word))

In [4]:
for x in range(100):
    print(df['Text'][x])
    print ('*******')

I want some chinamen hella bad I wish this coronavirus shit would die down a little 😂😩 damn
*******
@WhiteHouse House Dems want to pass a bill preventing  @realDonaldTrump  from enacting any travel ban at all because of the CoronaVirus. They called the China travel ban "xenophobic" and "racist".  Are they now going to call the EUROPE travel ban "racist"??  Dems have lost their minds!
*******
n ppl tried to come at me saying “stop calling it da chinese virus” when they created da shit even after governments stopped funding those dangerous ass trials. look stupid n defend da chinese if u want, they wanted dis to happen.
*******
@AliciaRedd2 @itsJeffTiedrich @realDonaldTrump I FUCK YOU baby, beautiful Only trump have guts to say CHINESE Virus and to take action against china. https://t.co/VTiOBn27zU
*******
@CoviDcreations @clif_high #coronavirus Report has been published first time and results are extremely serious.   Coronavirus has become the biggest challenge of 2020 in front of Scien

In [7]:
print(df['Text'][0])

I want some chinamen hella bad I wish this coronavirus shit would die down a little 😂😩 damn


In [177]:
print(df['Text'][1003])

The Asian delegation made a good draft pick in 2004.  #RacismIsAVirus https://t.co/NtMggc40WB


In [178]:
print(df['Text'][606])

Yaa, we won't call it Chinese.  @firkey_ @pokershash  #ChineseVirusCorona #ChineseVirus  #worldlockdown… https://t.co/r2hO6dyFOc


In [201]:
print(df['Text'][98])

Today I stand with my brothers and sisters of Asian descent to remind us that #RacismIsAVirus


In [172]:
print(df['Text'][20])

@dancezaizai @SteveTheDuckBoi @NineAttributes @RealSaavedra Keep eating bats, and rats, and dogs, and cats and tell us what a civilized country you come from. China owns this pandemic and the entire works knows it! #STFU #KungFlu #FU 🖕🏻 https://t.co/6YEhQRSTkw


In [26]:
# remove the 10 most frequent (but keep the first one!)

dellist = pd.Series(' '.join(df['Text']).split()).value_counts()[1:10]
print("COVID-HATE\n")
print(dellist)

COVID-HATE

to         1054
and         779
a           724
of          716
is          667
in          583
Chinese     394
you         394
for         334
dtype: int64


In [36]:
dellist_ = pd.Series(' '.join(df['Text']).split()).value_counts(ascending=True)[1:10]
print("COVID-HATE\n")
print(dellist_)

COVID-HATE

#Lockdown                  1
https://t.co/a6djq47XkD    1
want?                      1
(1/2)                      1
supremacist                1
@KamVTV                    1
https://t.co/TKfZctggLt    1
MSNBC,                     1
💖                          1
dtype: int64


# 3. Preprocessing

In [3]:
stop_words = set(stopwords.words('english'))
import contractions
from nltk.tokenize import word_tokenize
def clean_text(comment): 
   
    #comment = re.sub(r'[^a-zA-Z]', ' ',comment) 
    #comment = re.sub('[\\r\\t\\n]', ' ', comment)
    #comment = re.sub(' +',' ', comment)
    comment = re.sub("@\S+", " ", comment) # Remove usernames
    comment = re.sub("https*\S+", " ", comment) # Remove URL
    #comment = re.sub("\\<([^/> ]+)", " ", comment) # Remove HTML tags
    comment = comment.replace("#", "") #remode hashtag
    #comment = re.sub(r'[^a-zA-Z]', ' ',comment) 
    #comment = re.sub('[\\r\\t\\n]', ' ', comment)
    #comment = re.sub(' +',' ', comment)
    #comment = contractions.fix(comment)
    #months = {m.lower() for m in month_name[1:]}  # create a set of month names
    #for word in comment.split():
    #if word.lower() in months:
        #comment = comment.replace(word, "") 
   
    #word_tokens = word_tokenize(comment)
    # converts the words in word_tokens to lower case and then checks whether 
    #they are present in stop_words or not
    #filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
    #with no lower case conversion
    filtered_sentence = []

    #for w in word_tokens:
     #   if w not in stop_words:
      #      filtered_sentence.append(w)
  
 #   sentence = (" ").join(filtered_sentence)
    
    return comment

In [4]:
# clean and normalize comments
df['Text'] = df['Text'].map(lambda word:clean_text(word))

In [5]:
# remove the 10 most frequent (but keep the first one!)

dellist = pd.Series(' '.join(df['Text']).split()).value_counts(ascending=True)[1:100]
print("COVID-HATE\n")
print(dellist)

COVID-HATE

embodiedsolidarity    1
loosing               1
Bokhari               1
association           1
                     ..
throat.               1
many,                 1
SHIT..CHINA           1
communicate           1
injured               1
Length: 99, dtype: int64


In [8]:
dellist_ = pd.Series(' '.join(df['Text']).split()).value_counts(ascending=True)[41:100]
print("COVID-HATE\n")
print(dellist_)

COVID-HATE

+18                 1
F***                1
aborted             1
alone:              1
Jews:               1
dislike             1
hairy               1
SaudiArabia         1
Spain.              1
decisions.          1
taked               1
distracts           1
kits?               1
imo                 1
Human-To-Human      1
camps.              1
ships.              1
“common             1
Yep                 1
God's               1
AGENCY              1
goal                1
smuggle             1
Anything?           1
Frank               1
actions.            1
f…                  1
Find                1
snakes.             1
ilk                 1
TwitterKingSid      1
tweakin             1
TaiwanIsHelping     1
Tzi                 1
Lmao                1
$1,200              1
13th:               1
'As                 1
y'all!!!!!!!!!!!    1
angels              1
tanks.              1
profiling           1
Theyre              1
freetime            1
anymore.            

In [181]:
print(df['Text'][606])

Yaa, we won't call it Chinese.  @firkey_ @pokershash  ChineseVirusCorona ChineseVirus  worldlockdown… https://t.co/r2hO6dyFOc


In [188]:
print(df['Text'][1])

@whitehouse house dems want to pass a bill preventing  @realdonaldtrump  from enacting any travel ban at all because of the coronavirus. they called the china travel ban "xenophobic" and "racist".  are they now going to call the europe travel ban "racist"??  dems have lost their minds!


In [163]:
stop_words = set(stopwords.words('english'))
print(stop_words)

{'our', 'same', 'below', "hadn't", 't', 'those', 'off', "aren't", 'whom', 'other', 'will', 'wasn', 'up', "she's", 'what', 'it', 'your', 'why', 'couldn', 'theirs', 'more', "weren't", 'hadn', 'ain', 'i', 'should', "that'll", 'once', 'has', "won't", 'does', 'm', "it's", 'further', 'its', 'itself', "shouldn't", 'there', 'over', 'she', 'am', 'against', 'above', 'aren', 'yours', 'themselves', "should've", 'so', 'own', "couldn't", 'did', 'some', 'both', 'is', "you'll", 'not', 'shouldn', 'if', 'were', 'they', 'into', 'about', 'mightn', 'few', 'of', 'now', 'after', "didn't", 'but', "mustn't", 'him', 'than', 'which', 'ma', 'nor', 'doesn', 'no', 'having', 'because', 'my', 'with', 'any', "haven't", 'who', 'to', 'very', 'we', 'at', 'a', 'had', 'doing', 'hers', 'too', 'll', 'are', 'o', 'hasn', 'y', 'can', "you'd", 'you', 'being', 'her', 'on', 're', 'was', 'and', 'been', 'out', 'before', 'himself', 'ours', 'wouldn', 'where', 'here', 'ourselves', 'don', 'd', 'his', 'under', 'down', 'between', 'when', 

In [164]:
print(df['Text'][26])

Glenn pulled out all the stops for this one. #BigGovVirus https://t.co/KHC7mx6ziA


In [165]:
example_sent = (df['Text'][1003])
from nltk.tokenize import word_tokenize
word_tokens = word_tokenize(example_sent)
# converts the words in word_tokens to lower case and then checks whether 
#they are present in stop_words or not
filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
#with no lower case conversion
filtered_sentence = []
  
for w in word_tokens:
    if w not in stop_words:
        filtered_sentence.append(w)
  
print(word_tokens)
print(filtered_sentence)

['The', 'Asian', 'delegation', 'made', 'a', 'good', 'draft', 'pick', 'in', '2004', '.', '#', 'RacismIsAVirus', 'https', ':', '//t.co/NtMggc40WB']
['The', 'Asian', 'delegation', 'made', 'good', 'draft', 'pick', '2004', '.', '#', 'RacismIsAVirus', 'https', ':', '//t.co/NtMggc40WB']


In [166]:
sentence = (" ").join(filtered_sentence)
print(sentence)

The Asian delegation made good draft pick 2004 . # RacismIsAVirus https : //t.co/NtMggc40WB


In [50]:
from string import punctuation
print(punctuation)


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [32]:
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()

def lemmatization(text):
    """This function converts word to their root words 
       without explicitely cut down as done in stemming.
    
    arguments:
         input_text: "text" of type "String".
         
    return:
        value: Text having root words only, no tense form, no plural forms
        
    Example: 
    Input : text reduced 
    Output :  text reduce
    
   """
    # Converting words to their root forms
    lemma = [lemmatizer.lemmatize(w,'v') for w in w_tokenizer.tokenize(text)]
    return lemma

In [47]:
s =  (lemmatization("beautifulness"))

In [48]:
print(s)

['beautifulness']


In [34]:
sentence = (" ").join(s)
print(sentence)

You people love to step on other people, don't you?


In [59]:
import nltk
from nltk.stem import PorterStemmer
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
word_stemmer = PorterStemmer()
def stemer(text):
      # Converting words to their root forms
    s = [word_stemmer.stem(w,'v') for w in w_tokenizer.tokenize(text)]
    st = (" ").join(s)
    return st
text1 = 'at this point nigga be certify as water lmfaoooo he prolly the one that start that coronavirus 😂😂'
ss = stemer(text1)
#sentence = (" ").join(ss)
#print(sentence)


print(stemer(text1))

at thi point nigga be certifi as water lmfaoooo he prolli the one that start that coronaviru 😂😂


In [60]:
def stem(comment):
    comment= comment.lower()
    for word in comment.split():
        comment = comment.replace(word, word_stemmer.stem(word))
    return comment

In [61]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag


wordnet = WordNetLemmatizer()
def lemmatize_token1 (text):
    s = []
    text= text.lower()

    for token,tag in pos_tag(word_tokenize(text)):
        pos=tag[0].lower()
        
        if pos not in ['a', 'r', 'v', 's']:
            pos='n'
    
        s.append(wordnet.lemmatize(token,pos))
    
    sentence = (" ").join(s)
    return sentence


In [63]:
import contractions
import unicodedata
from spellchecker import SpellChecker
spelling = SpellChecker()

#spelling check
def check_spell(comment): 
    correct_words = []

    comment = comment.lower() # Normalize to lowercase 

    for w in comment.split():
        if w is None:
            continue                
        if len(w)>15 or len(w)==1:
            correct_words.append(w)
        else:
            correct_words.append(spelling.correction(w).lower())
    cor_comment = ' '.join(correct_words) 
    return cor_comment

In [64]:
# remove accented characters
def remove_accented_chars(text):
    new_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return new_text

In [68]:
text = '« Ya\'ll take note. There ain\'t never tornadoes where colored folk live we r nt stupid pathtic kids u r u fat shite '

import contractions
import unicodedata
comment = text.lower() # Normalize to lowercase 
comment = remove_accented_chars(comment) # remove accented characters
comment = contractions.fix(comment) # fix contractions
comment = lemmatize_token1(comment)
comment = stem(comment)
print(comment)

ya 'll take note . there be not never tornado where color folk live we r nt stupid pathtic kid you r you fat shite


In [70]:
def print_text(sample, clean):
    print(f"Before: {sample}")
    print(f"After: {clean}")

In [72]:
tokens = comment.split()
clean_tokens = [t for t in tokens if len(t) > 2]
clean_text = " ".join(clean_tokens)
print_text(comment, clean_text)


Before: ya 'll take note . there be not never tornado where color folk live we r nt stupid pathtic kid you r you fat shite
After: 'll take note there not never tornado where color folk live stupid pathtic kid you you fat shite


In [48]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV} # Pos tag, used Noun, Verb, Adjective and Adverb
# Function for lemmatization using POS tag
def lemmatize_words(text):
    pos_tagged_text = nltk.pos_tag(text.split())
    return " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text])
# Passing the function to 'text_rare' and store in 'text_lemma'
#df["text_lemma"] = df["text_rare"].apply(lemmatize_words)

In [67]:
s =  (lemmatization("aardwolves"))
sentence = (" ").join(s)
print(sentence)

aardwolves


In [72]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
print(wnl.lemmatize('dogs'))
#dog
print(wnl.lemmatize('churches'))
#church
print(wnl.lemmatize('aardwolves'))
#aardwolf
print(wnl.lemmatize('abaci'))
#abacus
print(wnl.lemmatize('hardrock'))
#hardrock
print(wnl.lemmatize('doing'))

dog
church
aardwolf
abacus
hardrock
doing


In [66]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

text = "Glenn pulled out all the stops for this one. #BigGovVirus https://t.co/KHC7mx6ziA"
wordnet = WordNetLemmatizer()
tokenizer = word_tokenize(text)
s = []

for token in tokenizer:
    s.append(wordnet.lemmatize(token))
sentence = (" ").join(s)
print(sentence)

Glenn pulled out all the stop for this one . # BigGovVirus http : //t.co/KHC7mx6ziA


In [12]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag

text1 = "This is such a dangerous thing to say and I’m disgusted how vile you are for posting this shit, you’re disgusting #ChinaLiedPeopleDied"
wordnet = WordNetLemmatizer()
def lemmatize_token1 (text):
    s = []

    for token,tag in pos_tag(word_tokenize(text)):
        pos=tag[0].lower()
        
        if pos not in ['a', 'r', 'v', 's']:
            pos='n'
    
        s.append(wordnet.lemmatize(token,pos))
    
    sentence = (" ").join(s)
    return sentence
print(lemmatize_token1(text1))

This be such a dangerous thing to say and I ’ m disgust how vile you be for post this shit , you ’ re disgust # ChinaLiedPeopleDied


In [129]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag

wordnet = WordNetLemmatizer()
def lemmatize_token (text):
    lemma = []

    for token,tag in pos_tag(word_tokenize(text)):
        pos = tag[0].lower()        
        if pos not in ['v', 'r', 'a', 's']:
            pos = 'n'    
        lemma.append(wordnet.lemmatize(token,pos))    
    sentence = (" ").join(s)
    return sentence

In [130]:
print(lemmatize_token(text1))

using returned sweetly aardwolf


In [26]:
print("Ensemble de données HatEval")
df.isnull().sum()

Ensemble de données HatEval


Tweet ID    0
Text        0
tox         0
dtype: int64

In [9]:
import datetime
start = '01MAY2017 11:45'
print (start_date.strftime('%d%b%Y %H:%M'))
start_date = datetime.datetime.strptime(start, '%d%b%Y %H:%M').date()
comment = '01MAY2017 11:45 hgwj fwkhw je'
comment = comment.replace('%d%b%Y %H:%M', " ")
print (comment)

01May2017 00:00
01MAY2017 11:45 hgwj fwkhw je


In [14]:
import re
found = ''
text = '2/17/1937 00:00:00:00'

m = re.search('(.*)\s[0\:]*$', text)
if m:
    found = m.group(1);
    print (found)

2/17/1937


In [24]:
from datetime_extractor import DateTimeExtractor
import re

samplestring1 = 'scala> val xorder= new order(1,"2016-02-22 00:00:00.00",100,"COMPLETED") Fri,  Jun ' 
y = DateTimeExtractor(samplestring1)
print(y)
for date in y:
    samplestring1 = samplestring1.replace(date, ' ')
print(samplestring1)

['2016-02-22 00:00:00.00']
scala> val xorder= new order(1," ",100,"COMPLETED") Fri,  Jun 


In [17]:
samplestring2 = 'Fri, 10 Jun 2011 11:04:17 +0200 (CEST)'

x = DateTimeExtractor(samplestring2)

['10 Jun 2011 11:04:17']